# Dataset Pipeline: Schema Validation with Pandera

This notebook mirrors the **Dataset Pipeline** guide at [learnmlops.ops4life.com/guides/dataset-pipeline/](https://learnmlops.ops4life.com/guides/dataset-pipeline/).

Define a Pandera schema that encodes the expected data contract for the employee attrition dataset, then run it against the ingested data. Validation failures are surfaced as structured error reports — not silent bad data downstream.

**What we'll cover:**
1. Load the ingested data
2. Define the Pandera schema
3. Run validation
4. Write `validated.csv`

In [ ]:
import pandas as pd

# pandera.pandas is the pandas-specific interface to the Pandera validation library.
# It provides DataFrameSchema, Column, and Check — the three building blocks
# of a data contract.
import pandera.pandas as pa
from pandera import Column, Check


## Step 1: Load ingested data

In [ ]:
df = pd.read_csv('/workspace/pipeline-data/raw_ingested.csv')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(3)

## Step 2: Define the Pandera schema

In [ ]:
employee_schema = pa.DataFrameSchema(
    {
        # Column(dtype, check, nullable) — three things to enforce per column:
        #   dtype:    the expected Python/pandas type (int, str, float, etc.)
        #   check:    a value constraint — Check.between, .gt, .isin, etc.
        #   nullable: whether NaN is allowed (False = reject any row with NaN here)

        'Age':             Column(int, Check.between(22, 59), nullable=False),  # Working-age range
        'Attrition':       Column(str, Check.isin(['Yes', 'No']), nullable=False),
        'BusinessTravel':  Column(str, Check.isin(['Non-Travel', 'Travel_Rarely', 'Travel_Frequently']), nullable=False),
        'DailyRate':       Column(int, Check.gt(0), nullable=False),   # Must be positive (gt = greater than)
        'Department':      Column(str, nullable=False),                 # No value constraint — any string is valid
        'DistanceFromHome': Column(int, Check.ge(1), nullable=False),  # ge = greater than or equal to
        'Education':       Column(int, Check.between(1, 4), nullable=False),

        # unique=True rejects duplicate values — EmployeeNumber is a primary key.
        'EmployeeNumber':  Column(int, unique=True, nullable=False),

        'EnvironmentSatisfaction': Column(int, Check.between(1, 3), nullable=False),
        'Gender':          Column(str, Check.isin(['Male', 'Female']), nullable=False),
        'HourlyRate':      Column(int, Check.gt(0), nullable=False),
        'JobInvolvement':  Column(int, Check.between(1, 3), nullable=False),
        'JobLevel':        Column(int, Check.between(1, 4), nullable=False),
        'JobRole':         Column(str, nullable=False),
        'JobSatisfaction': Column(int, Check.between(1, 3), nullable=False),
        'MaritalStatus':   Column(str, Check.isin(['Single', 'Married', 'Divorced']), nullable=False),
        'MonthlyIncome':   Column(int, Check.gt(0), nullable=False),
        'OverTime':        Column(str, Check.isin(['Yes', 'No']), nullable=False),

        # PerformanceRating only has values 3 or 4 in the IBM dataset — 1 and 2 never appear.
        # Using Check.isin (not Check.between) enforces this exact constraint.
        'PerformanceRating': Column(int, Check.isin([3, 4]), nullable=False),

        'RelationshipSatisfaction': Column(int, Check.between(1, 3), nullable=False),
        'TotalWorkingYears': Column(int, Check.ge(0), nullable=False),  # 0 = new hire
        'WorkLifeBalance': Column(int, Check.between(1, 3), nullable=False),
        'YearsAtCompany':  Column(int, Check.ge(0), nullable=False),
        'YearsInCurrentRole': Column(int, Check.ge(0), nullable=False),
        'YearsSinceLastPromotion': Column(int, Check.ge(0), nullable=False),
        'YearsWithCurrManager': Column(int, Check.ge(0), nullable=False),
    },
    # strict=False: columns not listed in the schema are silently ignored.
    # strict=True would raise an error for any extra column — useful if you want
    # an exact schema contract and no surprise columns from upstream.
    strict=False,

    # coerce=True: cast each column to its declared dtype before running checks.
    # Example: if a CSV loads 'Age' as float64 (e.g. 45.0), coerce converts it to int.
    # Without coerce, the dtype check would fail on any such mismatch.
    coerce=True,
)

print('Schema defined with', len(employee_schema.columns), 'column checks')


Pandera validates each column's dtype and value constraints before data moves forward in the pipeline. `strict=False` allows extra columns in the data. `coerce=True` casts columns to their declared dtype before checking — catching type mismatches that would otherwise cause silent downstream errors.

## Step 3: Run validation

In [ ]:
def validate_data(df):
    try:
        # lazy=True collects ALL validation errors before raising an exception.
        # Without lazy=True, Pandera stops at the first failure — you'd have to
        # fix one error, re-run, find the next, and so on.
        # With lazy=True, you get a complete list of all failures in one pass.
        validated_df = employee_schema(df, lazy=True)
        print('Validation passed.')
        return validated_df

    except pa.errors.SchemaErrors as e:
        # e.failure_cases is a DataFrame with columns:
        #   schema_context — which schema element failed (Column, Index, etc.)
        #   column         — the column name
        #   check          — the check that failed (e.g. "between(22, 59)")
        #   check_number   — index into the column's check list
        #   failure_case   — the actual value that violated the check
        #   index          — the row index where the failure occurred
        # This structured output makes it easy to trace bad data back to its source.
        print('Validation errors:')
        print(e.failure_cases)
        return None

validated_df = validate_data(df)


If validation passes, `validate_data()` returns the coerced DataFrame. If any column has values outside the declared range, Pandera prints a structured table of failures (column name, check name, failure value, row index) — making it easy to trace bad data back to its source.

## Step 4: Write `validated.csv`

In [ ]:
output_path = '/workspace/pipeline-data/validated.csv'
validated_df.to_csv(output_path, index=False)
print(f'Written: {output_path}')
print(f'Shape: {validated_df.shape}')

## Next Steps

- Continue to feature engineering: `02-pipeline/data-preparation/feature_engineering.ipynb`
- Or see the Airflow DAG that orchestrates all pipeline steps: `02-pipeline/dataset-pipeline/dag.ipynb`
- Full guide: [learnmlops.ops4life.com/guides/dataset-pipeline/](https://learnmlops.ops4life.com/guides/dataset-pipeline/)